In [1]:
import maboss
import scanpy as sc
import liana as li
from pathlib import Path
import os

cwd=Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent) 
os.getcwd()

'/home/maxi7524/repositories/project_computational_biology_initial_conditions_pymyboss'

## Loading part


## Remarks 

### loading resources

Here i load RNA data, important is that we need to:
- use proper resource for LIG/REC pairs tokens 
- select proper in aggregate functions (many samples does met criterias )

In [2]:
PATH_data = Path('data/multimodal')
rna = sc.read(PATH_data / "sma_rna.h5ad")
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

# important mouse data - for token genes
mouse_resource = li.rs.select_resource(resource_name='mouseconsensus')

In [7]:
rna.layers['counts']

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 6297195 stored elements and shape (3036, 16486)>

In [5]:
rna.obs

,in_tissue,array_row,array_col,x,y,lesion,region,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,n_genes,n_counts
AAACAAGTATCTCCCA-1,1,50,102,34740,30651,lesioned,striatum,1562,7.354362,3491.0,8.158230,38.556288,44.714981,53.566313,69.578917,1171.0,7.066467,33.543396,1562,3491.0
AAACAGCTTTCAGAAG-1,1,43,9,7784,27046,intact,not_striatum,626,6.440947,1454.0,7.282761,55.295736,62.242091,70.701513,91.334250,632.0,6.450470,43.466301,626,1454.0
AAACATTTCCCGGATT-1,1,61,97,33275,36197,lesioned,striatum,2275,7.730175,5784.0,8.663024,37.672891,43.533887,51.175657,65.058783,1978.0,7.590347,34.197788,2275,5784.0
AAACCCGAACGAAATC-1,1,45,115,38516,28139,lesioned,not_striatum,1193,7.085064,2048.0,7.625107,29.541016,37.353516,47.656250,66.162109,501.0,6.218600,24.462891,1193,2048.0
AAACCGGAAATGTTAA-1,1,54,124,41113,32686,lesioned,not_striatum,1127,7.028201,1846.0,7.521318,28.169014,36.294691,47.345612,66.034670,406.0,6.008813,21.993500,1127,1846.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTCAGTGTGCTAC-1,1,24,64,23757,17504,intact,not_striatum,2813,7.942362,6917.0,8.841882,29.044383,34.841694,42.706376,57.683967,1871.0,7.534763,27.049299,2813,6917.0
TTGTTCTAGATACGCT-1,1,21,3,6074,15942,intact,not_striatum,2473,7.813592,5693.0,8.647168,27.191288,33.971544,43.052872,59.195503,1362.0,7.217443,23.924116,2473,5693.0
TTGTTGTGTGTCAAGA-1,1,31,77,27517,21045,lesioned,not_striatum,2835,7.950150,7132.0,8.872487,31.015143,37.268648,45.288839,59.632642,2022.0,7.612337,28.351093,2835,7132.0
TTGTTTCCATACAACT-1,1,45,27,13000,28069,intact,striatum,2751,7.920083,8027.0,8.990691,42.319671,47.439890,53.942943,66.126822,3069.0,8.029433,38.233463,2751,8027.0


In [4]:
# Run rank_aggregate
li.mt.rank_aggregate(rna, 
                     groupby='region',
                     resource=mouse_resource,
                     expr_prop=0.1,
                     use_raw = False,           # we change default layer
                     layer='counts',            # we apply specific layer
                     verbose=True)

Using provided `resource`.


Using the `counts` layer!


/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
Make sure that normalized counts are passed!
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.27 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 3036 samples and 1079 features


/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:293: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:296: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [00:02<00:00, 395.79it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


In [5]:
rna.uns.keys()

dict_keys(['lesion_colors', 'log1p', 'region_colors', 'spatial', 'liana_res'])

In [6]:
from MaBossSpatial import SpatialBooleanPipeline 

In [7]:
# Setting up pipeline 

## initialization
pipeline = SpatialBooleanPipeline(
    ## adata file, REMARK: it need to contain liana results
    spatial_data=rna, 
    ## value with liana results, you can find it in adata.uns.keys()
    liana_uns_key='liana_res' 
) 

## loading PyMyBoss model
pipeline.SetMaBossModel(
    mode='pretrained',
    model_name='Creative_name',
    bnd_path='data/maboss/CellFate/CellFateModel.bnd',
    cfg_path='data/maboss/CellFate/CellFateModel.cfg'
)

pipeline.SetSpatialSettings(bandwidth=30.0, cutoff=0.05, kernel="gaussian")

pipeline.SetTimeLags(strategy="experimental", custom_lags={"TNF": 10.0, "FAS": 15.0})

pipeline.SetSimulationSettings(max_time=45.0, delta_t=5.0, sample_count=1000)

Successfully loaded pretrained MaBoSS model 'Creative_name' with 28 nodes.


In [8]:
target_cells = rna.obs_names[:2].tolist()
output_path = "data/maboss/real_simulation_results.csv"

pipeline.RunPipeline(target_cell_ids=target_cells, output_csv_path=output_path)

Computing spatial neighborhood connectivity matrix via LIANA+...
Evaluating biological time lags for active pathways...
                    SPATIAL BOOLEAN PIPELINE CONFIGURATION REPORT

[ COMPATIBILITY ALERT ]
--------------------------------------------------------------------------------
⚠️  Warning: Nomenclature Mismatch Detected!
    -> Intracellular Model Organism : HUMAN (Detected via UPPERCASE nodes)
    -> Spatial Expression Dataset   : MOUSE (Detected via Sentence-Case genes)
    -> Impact: Continuous mapping fields will register 0.0 unless matching case rules are added.

--- 1. MODEL TOPOLOGY & MAPPING STATUS ---
    Total Managed Network Nodes: 28
    Directly Mapped Gene Overlaps: 0
    ⚠️  No precise node-to-gene name mapping detected. Check capitalizations.

--- 2. SPATIAL & COMMUNICATION LAYER ---
    ✅  LIANA+ Connectivity Key   : spatial_connectivities
    ✅  Active Spatial Edges      : 3036 connections resolved in tissue grid.

--- 3. TEMPORAL & TIME-LAG CONFIGURATIO

In [10]:
"""
Diagnostic script to validate data mapping, spatial connectivity, and lag structures
within the loaded MaBossSpatial pipeline.
"""
print("=== KROK 1: Weryfikacja mapowania nazw genów ===")
model_nodes = pipeline.maboss_manager.all_nodes
matched_genes = [node for node in model_nodes if node in pipeline.adata.var_names]
missing_genes = [node for node in model_nodes if node not in pipeline.adata.var_names]

print(f"Wszystkie węzły modelu ({len(model_nodes)}): {model_nodes}")
print(f"Dopasowane geny znalezione w adata ({len(matched_genes)}): {matched_genes}")
print(f"Węzły nieobecne w adata (otrzymają warunek początkowy 0.0): {missing_genes}")

print("\n=== KROK 2: Weryfikacja grafu przestrzennego LIANA+ ===")
conn_matrix = pipeline.adata.obsp[pipeline.spatial_env.connectivity_key]
total_edges = conn_matrix.count_nonzero() if hasattr(conn_matrix, "count_nonzero") else np.count_nonzero(conn_matrix)
print(f"Macierz sąsiedztwa LIANA+ zawiera: {total_edges} aktywnych połączeń przestrzennych między komórkami.")

print("\n=== KROK 3: Weryfikacja konfiguracji lagów czasowych ===")
if pipeline.time_estimator and pipeline.time_estimator.intracellular_lags:
    print("Zarejestrowane wewnątrzkomórkowe opóźnienia dla receptorów:")
    for receptor, lag in pipeline.time_estimator.intracellular_lags.items():
        print(f"  - Receptor: {receptor} -> Lag: {lag} minut")
else:
    print("UWAGA: Brak wyliczonych lagów dla receptorów. Sprawdź, czy nazwy w liana_res pokrywają się z modelem.")

=== KROK 1: Weryfikacja mapowania nazw genów ===
Wszystkie węzły modelu (28): ['ATP', 'Apoptosis', 'BAX', 'BCL2', 'CASP3', 'CASP8', 'Cyt_c', 'DISC_FAS', 'DISC_TNF', 'FADD', 'FASL', 'IKK', 'MOMP', 'MPT', 'NFkB', 'NonACD', 'RIP1', 'RIP1K', 'RIP1ub', 'ROS', 'SMAC', 'Survival', 'TNF', 'TNFR', 'XIAP', 'apoptosome', 'cFLIP', 'cIAP']
Dopasowane geny znalezione w adata (0): []
Węzły nieobecne w adata (otrzymają warunek początkowy 0.0): ['ATP', 'Apoptosis', 'BAX', 'BCL2', 'CASP3', 'CASP8', 'Cyt_c', 'DISC_FAS', 'DISC_TNF', 'FADD', 'FASL', 'IKK', 'MOMP', 'MPT', 'NFkB', 'NonACD', 'RIP1', 'RIP1K', 'RIP1ub', 'ROS', 'SMAC', 'Survival', 'TNF', 'TNFR', 'XIAP', 'apoptosome', 'cFLIP', 'cIAP']

=== KROK 2: Weryfikacja grafu przestrzennego LIANA+ ===
Macierz sąsiedztwa LIANA+ zawiera: 3036 aktywnych połączeń przestrzennych między komórkami.

=== KROK 3: Weryfikacja konfiguracji lagów czasowych ===
Zarejestrowane wewnątrzkomórkowe opóźnienia dla receptorów:
  - Receptor: Lrp1 -> Lag: 10.0 minut
  - Receptor